ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape
# Output: (8950, 18) — 8950 customers and 18 columns

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()
# CREDIT_LIMIT has 1 missing value
# MINIMUM_PAYMENTS has 313 missing values

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()
# All columns should now show 0 missing values

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(20, 16), bins=30, edgecolor='black', color='steelblue')
plt.suptitle('Distribution of Credit Card Features', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap of Credit Card Features', fontsize=14)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, color='royalblue', edgecolors='none', s=10)
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, color='tomato', edgecolors='none', s=10)
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue', linewidth=2, markersize=7)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal K for Credit Card Clustering')
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

**Output Interpretation**

The inertia drops steeply from K=1 to K=3, and then starts to slow down noticeably after K=3. The curve bends around **K=3**, suggesting that 3 clusters captures most of the structure in the data without overfitting.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), silhouette_scores, marker='s', color='darkorange', linewidth=2, markersize=7)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(range(2, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_df = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': [round(s, 4) for s in silhouette_scores]
})

silhouette_df.set_index('K', inplace=True)
silhouette_df

**Output Interpretation**

The highest silhouette score is at **K=3 (≈ 0.2506)**, meaning the clusters are most clearly separated at this K value. While the scores are generally low (common with high-dimensional data), K=3 consistently provides the best separation and also aligns with the elbow point — confirming it as the best choice.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# K=3 is chosen because:
# 1. The elbow curve shows a significant bend at K=3
# 2. K=3 achieves the highest silhouette score (0.2506)
# 3. 3 distinct customer segments are interpretable for business use

final_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
final_kmeans.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_kmeans.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
cluster_counts = df['Cluster'].value_counts().sort_index().reset_index()
cluster_counts.columns = ['Cluster', 'Customer Count']
print(cluster_counts)

# Visualize cluster sizes
plt.figure(figsize=(7, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
plt.bar(cluster_counts['Cluster'], cluster_counts['Customer Count'],
        color=colors, edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Number of Customers per Cluster')
plt.xticks([0, 1, 2])
for i, row in cluster_counts.iterrows():
    plt.text(row['Cluster'], row['Customer Count'] + 30, str(row['Customer Count']),
             ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)

# Fit PCA on the scaled data (without the Cluster column)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PCA Component 1', 'PCA Component 2'])
pca_df['Cluster'] = final_kmeans.labels_

plt.figure(figsize=(10, 7))
colors = ['#4C72B0', '#DD8452', '#55A868']
labels = ['Cluster 0: Cash Advance Users', 'Cluster 1: High-Value Active Buyers', 'Cluster 2: Low Engagement']

for cluster_id, color, label in zip([0, 1, 2], colors, labels):
    subset = pca_df[pca_df['Cluster'] == cluster_id]
    plt.scatter(subset['PCA Component 1'], subset['PCA Component 2'],
                c=color, label=label, alpha=0.5, s=12, edgecolors='none')

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('K-Means Clustering of Credit Card Customers (PCA 2D View)')
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

explained = pca.explained_variance_ratio_
print(f"Explained variance: PC1 = {explained[0]:.2%}, PC2 = {explained[1]:.2%}, Total = {sum(explained):.2%}")

**Output Interpretation**

The PCA plot shows a 2D projection of the clusters. The three groups are partially separated — Cluster 1 (high-value active buyers) forms the most distinct region, while Clusters 0 and 2 overlap more because they share some behavioral similarities. This overlap is expected since PCA only captures a portion of the total variance (the first two components explain around 33% of the variation).

## Final Questions

### 1. Why is this an unsupervised learning problem?

This is an unsupervised learning problem because the dataset has **no target label** or predefined customer group column. We are not predicting a known outcome — instead, we are discovering hidden patterns in the data. K-Means finds natural groupings based purely on the behavioral features, without any guidance from labeled examples.

---

### 2. Why did we remove the `CUST_ID` column?

`CUST_ID` is a unique identifier for each customer. It carries no behavioral information about how the customer uses their credit card. Including it would confuse the clustering algorithm because K-Means would try to measure distances between arbitrary ID codes that have no real numeric meaning.

---

### 3. Which columns had missing values?

Two columns had missing values:
- **`CREDIT_LIMIT`** — 1 missing value
- **`MINIMUM_PAYMENTS`** — 313 missing values

---

### 4. How did you handle the missing values?

We used **mean imputation** — replacing each missing value with the column's mean. This is appropriate here because the missing values are a small fraction of the total data and the columns represent continuous financial values where the average is a reasonable estimate.

---

### 5. Why is scaling important before applying K-Means?

K-Means calculates **Euclidean distances** between data points. Features with large numerical ranges (like `BALANCE` or `CREDIT_LIMIT`, which can be in the thousands) would dominate the distance calculation over features with small ranges (like `PURCHASES_FREQUENCY`, which is between 0 and 1). StandardScaler transforms all features to have **mean = 0 and standard deviation = 1**, so every feature contributes equally to the clustering.

---

### 6. Which K value did you choose? Explain your reasoning.

We chose **K = 3**, based on two pieces of evidence:

- **Elbow Method**: The inertia curve shows a clear bend (elbow) at K=3. After K=3, the decrease in inertia becomes much more gradual, meaning additional clusters add little improvement.
- **Silhouette Score**: K=3 achieved the **highest silhouette score of ~0.2506**, indicating that at K=3 the clusters are most cohesive and well-separated compared to other K values tested.

Both methods converge on K=3, and 3 segments is also a meaningful number from a business standpoint.

---

### 7. Describe each customer segment based on the cluster summary table.

| Cluster | Customers | Key Characteristics | Segment Name |
|---------|-----------|---------------------|--------------|
| **0** | 1,596 | High balance (~$3,989), very high cash advance (~$3,870), very low purchases, low purchase frequency | **Cash Advance Reliant Users** |
| **1** | 1,235 | Moderate balance (~$2,220), very high purchases (~$4,269), high purchase frequency (0.95), high credit limit (~$7,734), high payments | **High-Value Active Buyers** |
| **2** | 6,119 | Low balance (~$800), low purchases (~$506), very low cash advance (~$330), low credit limit (~$3,270) | **Low Engagement / Passive Users** |

---

### 8. Which cluster may represent high-value customers?

**Cluster 1** represents high-value customers. These customers have the highest purchase amounts (~$4,269 average), the highest purchase frequency (0.95 — meaning they use their card almost constantly), the highest credit limits (~$7,735), and the highest payments (~$4,151). They are actively and responsibly using their credit cards, making them the most valuable segment for the bank.

---

### 9. Which cluster may represent customers who rely more on cash advance?

**Cluster 0** relies most heavily on cash advances. The average cash advance amount is ~$3,870 and the cash advance frequency is 0.45 (much higher than other clusters). These customers carry high balances but make very few purchases, suggesting they use their credit card primarily as a source of emergency liquidity rather than for everyday spending.

---

### 10. How can a company use these clusters for marketing strategy?

| Cluster | Marketing Strategy |
|---------|-------------------|
| **Cluster 0** (Cash Advance Users) | Offer lower cash advance interest rates or personal loan products as a safer alternative. Provide financial wellness resources to reduce dependency on cash advances. |
| **Cluster 1** (High-Value Buyers) | Reward with premium loyalty programs, cashback offers, and higher credit limits. Upsell premium cards (travel rewards, concierge services). Retain them with exclusive perks. |
| **Cluster 2** (Low Engagement) | Run re-engagement campaigns with spending incentives (e.g., bonus cashback on first purchases). Educate them on card benefits. Offer introductory low-interest rates to encourage more usage. |